# Pilot v1 Analysis — 5-condition multi-select

Conditions: `birds` (control), `birds_easier`, `birds_listed`, `birds_distractors`, `birds_repeat`  
Scoring: all-or-nothing per question (primary) + per-statement Hamming (secondary)

In [ ]:
import csv
import re
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import yaml

HERE      = Path('.')  # run from pilot_v1/
CSV_PATH  = HERE / 'QA+Pilot+v2_June+15,+2026_12.17.csv'
YAML_PATH = HERE.parent / 'birds_q_multiselect.yaml'

# Q01..Q10 columns map 1-to-1 to QS01..QS10 in the YAML
QCOLS      = [f'Q{n:02d}' for n in range(1, 11)]
QIDS       = [f'QS{n:02d}' for n in range(1, 11)]
COL_TO_QID = dict(zip(QCOLS, QIDS))
LETTERS    = list('ABCDE')

CONDITIONS = ['birds', 'birds_easier', 'birds_listed', 'birds_distractors', 'birds_repeat']
COND_COLORS = {
    'birds':             '#1f77b4',
    'birds_easier':      '#2ca02c',
    'birds_listed':      '#ff7f0e',
    'birds_distractors': '#d62728',
    'birds_repeat':      '#9467bd',
}
COND_LABELS = {c: c.replace('birds_', '') for c in CONDITIONS}
COND_LABELS['birds'] = 'control'

# Attention check correct answers (used to exclude failed respondents)
AT1_CORRECT = 'Green versus rose'
AT2_CORRECT = 'Plumage strategy'

In [ ]:
# Load YAML and CSV
spec    = yaml.safe_load(YAML_PATH.read_text())
by_qid  = {q['q_id']: q for q in spec['questions']}

with CSV_PATH.open(newline='', encoding='utf-8-sig') as f:
    rows = list(csv.DictReader(f))

all_finished = [r for r in rows[2:] if r.get('Finished', '').lower() in {'true', '1'}]
data_rows = [
    r for r in all_finished
    if r.get('PROLIFIC_PID', '').strip()
    and r.get('AT1', '').strip() == AT1_CORRECT
    and r.get('AT2', '').strip() == AT2_CORRECT
]

print(f'Finished: {len(all_finished)}, kept after exclusions: {len(data_rows)}')
print('By condition:', Counter(r['assigned_doc'] for r in data_rows))

In [ ]:
OPTION_ALIASES = {
    ('QS06', 'B'): [
        'The crestling has a retractable cartilaginous crest that flushes a faint rose colour from engorged blood vessels.',
    ],
    ('QS08', 'B'): ['The crestling is roughly 18 cm from head to tail.'],
    ('QS08', 'D'): ['The sunwhistle is larger than the crestling.'],
}

def parse_selected(response_text, options_dict, qid=None, strict=False):
    """
    Map Qualtrics comma-joined response text to a set of option letters.

    Normalizations:
      - ' -- ' → ' - '  (Qualtrics double-dash vs YAML single dash, affects QS02)
      - Strip markdown underscores and trailing period from option key

    OPTION_ALIASES covers survey option text that changed after YAML update.
    strict=True raises ValueError if a non-empty response matches no option.
    """
    if not response_text or not response_text.strip():
        return set()
    resp = re.sub(r' -- ', ' - ', response_text.strip())
    selected = set()
    for letter, opt_text in options_dict.items():
        key = re.sub(r'_', '', opt_text.strip()).rstrip('.')
        key = re.sub(r' -- ', ' - ', key)
        if key in resp:
            selected.add(letter)
        if qid:
            for alias in OPTION_ALIASES.get((qid, letter), []):
                if re.sub(r'_', '', alias.strip()).rstrip('.') in resp:
                    selected.add(letter)
    if strict and not selected:
        raise ValueError(
            f'[{qid}] Response matched no option.\n'
            f'  Response: {repr(response_text[:120])}\n'
            f'  Options:  {list(options_dict.values())}'
        )
    return selected


def all_or_nothing(selected, answer_key):
    return int(set(selected) == set(answer_key))


def hamming_acc(selected, answer_key, n_opts=5):
    key_set = set(answer_key)
    sel_set = set(selected)
    return sum(1 for L in LETTERS[:n_opts] if (L in sel_set) == (L in key_set)) / n_opts


respondent_data = []
for r in data_rows:
    rec = {
        'cond': r['assigned_doc'],
        'pid':  (r.get('PROLIFIC_PID') or r.get('ResponseId', ''))[:12],
    }
    for qcol, qid in COL_TO_QID.items():
        q   = by_qid[qid]
        sel = parse_selected(r.get(qcol, ''), q['options'], qid=qid, strict=True)
        rec[qid] = {
            'selected': sel,
            'aon':      all_or_nothing(sel, q['answer']),
            'hamming':  hamming_acc(sel, q['answer']),
        }
    rec['total_aon']     = sum(rec[qid]['aon']     for qid in QIDS)
    rec['total_hamming'] = np.mean([rec[qid]['hamming'] for qid in QIDS])
    respondent_data.append(rec)

PARAPHRASE = {qid for qid in QIDS if by_qid[qid]['metadata'].get('cue_match') == 'paraphrase'}

print(f'Parsed {len(respondent_data)} respondents — strict mode passed.')
print(f'Paraphrase questions: {PARAPHRASE}')

In [ ]:
# ── Answer distributions: one file per question ───────────────────────────────
# Saves answer_dist_QS01.png … answer_dist_QS10.png

def wrap(s, n=22):
    s = re.sub(r'_', '', s.strip())
    words, lines, line = s.split(), [], ''
    for w in words:
        if len(line) + len(w) + 1 > n: lines.append(line); line = w
        else: line = (line + ' ' + w).strip()
    if line: lines.append(line)
    return '\n'.join(lines)

for qcol, qid in COL_TO_QID.items():
    q       = by_qid[qid]
    opts    = q['options']
    ans_set = set(q['answer'])
    letters = LETTERS[:len(opts)]
    x       = np.arange(len(letters))
    star    = '*' if qid in PARAPHRASE else ''

    fig, ax = plt.subplots(figsize=(10, 5))

    for i, L in enumerate(letters):
        if L in ans_set:
            ax.axvspan(i - 0.5, i + 0.5, color='#fffacd', zorder=0)

    for ci, cond in enumerate(CONDITIONS):
        recs  = [rec for rec in respondent_data if rec['cond'] == cond]
        n     = len(recs)
        rates = [sum(1 for rec in recs if L in rec[qid]['selected']) / n * 100 if n else 0
                 for L in letters]
        ax.bar(x + offsets[ci], rates, width=bar_w * 0.9,
               color=COND_COLORS[cond], label=f'{COND_LABELS[cond]} (N={n})', zorder=2)

    ax.set_title(f'{qid}{star} ({qcol})  |  answer: {q["answer"]}\n{q["question"]}',
                 fontsize=10, loc='left')
    ax.set_xticks(x)
    ax.set_xticklabels([f'{L}\n{wrap(opts[L])}' for L in letters], fontsize=8)
    ax.set_ylim(0, 110)
    ax.set_ylabel('% selected', fontsize=9)
    ax.tick_params(axis='y', labelsize=8)
    ax.legend(fontsize=8, ncol=n_cond, loc='upper right')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.text(0.01, 0.97, 'yellow = correct option  |  * = paraphrase cue-match',
            transform=ax.transAxes, fontsize=7, va='top', color='#555')

    plt.tight_layout()
    fig.savefig(f'answer_dist_{qid}.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved answer_dist_{qid}.png')

In [ ]:
# ── Correctness grids: one consolidated PNG ───────────────────────────────────
# 1 row × 5 cols. Rows sorted by total_aon desc. * = paraphrase.

from matplotlib.colors import ListedColormap
cmap_pad = ListedColormap(['#ffffff', '#eeeeee', '#2ca02c'])

max_n = max(len([rec for rec in respondent_data if rec['cond'] == c]) for c in CONDITIONS)
fig, axes = plt.subplots(1, 5, figsize=(22, max(4, 0.42 * max_n)))

for ax, cond in zip(axes, CONDITIONS):
    recs = sorted(
        [rec for rec in respondent_data if rec['cond'] == cond],
        key=lambda r: -r['total_aon']
    )
    n = len(recs)
    if n == 0:
        ax.axis('off')
        continue

    M        = np.array([[rec[qid]['aon'] for qid in QIDS] for rec in recs])
    row_accs = M.mean(axis=1)
    col_accs = M.mean(axis=0)
    pids     = [r['pid'] or f'R{i}' for i, r in enumerate(recs)]

    pad   = np.full((max_n - n, len(QIDS)), -1) if n < max_n else np.empty((0, len(QIDS)))
    M_pad = np.vstack([M, pad]) if len(pad) else M

    ax.imshow(M_pad, aspect='auto', cmap=cmap_pad, vmin=-1, vmax=1, interpolation='nearest')

    xlabels = [f'{qid}{"*" if qid in PARAPHRASE else ""}\n({a:.0%})' for qid, a in zip(QIDS, col_accs)]
    ax.set_xticks(range(len(QIDS)))
    ax.set_xticklabels(xlabels, fontsize=6.5)
    ylabels = [f'{p} {a:.0%}' for p, a in zip(pids, row_accs)] + [''] * max(0, max_n - n)
    ax.set_yticks(range(max_n))
    ax.set_yticklabels(ylabels, fontsize=6)
    ax.set_title(f'{COND_LABELS[cond]}\n(N={n})', fontsize=9, color=COND_COLORS[cond], fontweight='bold')

    ax.set_xticks(np.arange(-0.5, len(QIDS), 1), minor=True)
    ax.set_yticks(np.arange(-0.5, max_n, 1), minor=True)
    ax.grid(which='minor', color='white', linewidth=1)
    ax.tick_params(which='minor', length=0)

plt.suptitle('Correctness grids by condition  |  green = correct (all-or-nothing)  |  * = paraphrase',
             fontsize=10, y=1.01)
plt.tight_layout()
fig.savefig('correctness_grids.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved correctness_grids.png')

In [ ]:
# ── Score comparison: faceted by condition ────────────────────────────────────
# 5 rows × 2 cols.  Each row = one condition; col 0 = AoN, col 1 = Hamming.

fig, axes = plt.subplots(5, 2, figsize=(10, 16), sharey=False,
                         sharex='col')

bins_aon = np.arange(-0.5, len(QIDS) + 1.5, 1)
bins_ham = np.linspace(0.4, 1.01, 13)

for row, cond in enumerate(CONDITIONS):
    recs   = [rec for rec in respondent_data if rec['cond'] == cond]
    n      = len(recs)
    color  = COND_COLORS[cond]
    label  = COND_LABELS[cond]

    aon_scores = [rec['total_aon']     for rec in recs]
    ham_scores = [rec['total_hamming'] for rec in recs]

    # Col 0 — all-or-nothing
    ax = axes[row, 0]
    ax.hist(aon_scores, bins=bins_aon, color=color, alpha=0.85)
    ax.axvline(np.mean(aon_scores), color='black', linestyle='--', linewidth=1.2)
    ax.set_ylabel(f'{label}\n(N={n})', fontsize=9, color=color, fontweight='bold')
    ax.text(0.97, 0.95, f'μ={np.mean(aon_scores):.1f}', transform=ax.transAxes,
            ha='right', va='top', fontsize=8)
    if row == 0:
        ax.set_title('All-or-nothing (of 10)', fontsize=9)
    if row == len(CONDITIONS) - 1:
        ax.set_xlabel('# correct', fontsize=8)
    ax.set_xlim(-0.5, len(QIDS) + 0.5)

    # Col 1 — Hamming
    ax = axes[row, 1]
    ax.hist(ham_scores, bins=bins_ham, color=color, alpha=0.85)
    ax.axvline(np.mean(ham_scores), color='black', linestyle='--', linewidth=1.2)
    ax.text(0.97, 0.95, f'μ={np.mean(ham_scores):.3f}', transform=ax.transAxes,
            ha='right', va='top', fontsize=8)
    if row == 0:
        ax.set_title('Per-statement Hamming accuracy', fontsize=9)
    if row == len(CONDITIONS) - 1:
        ax.set_xlabel('Accuracy', fontsize=8)

fig.suptitle('Score distributions by condition  |  dashed = mean', fontsize=11, y=1.01)
plt.tight_layout()
fig.savefig('score_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved score_comparison.png')

In [ ]:
# ── Per-question accuracy and Hamming: grouped bar plots ─────────────────────

def bootstrap_ci(vals, n_boot=2000, ci=95, rng=None):
    if rng is None:
        rng = np.random.default_rng(0)
    vals = np.asarray(vals, dtype=float)
    if len(vals) == 0:
        return 0.0, 0.0
    boot_means = np.array([rng.choice(vals, size=len(vals), replace=True).mean()
                           for _ in range(n_boot)])
    lo = np.percentile(boot_means, (100 - ci) / 2)
    hi = np.percentile(boot_means, 100 - (100 - ci) / 2)
    return vals.mean() - lo, hi - vals.mean()

rng_boot = np.random.default_rng(42)
n_cond   = len(CONDITIONS)
bar_w    = 0.7 / n_cond
offsets  = np.linspace(-(n_cond - 1) / 2, (n_cond - 1) / 2, n_cond) * bar_w
x        = np.arange(len(QIDS))

def bar_plot(metric, ylabel, fname, ylim, title, scale=1.0):
    fig, ax = plt.subplots(figsize=(14, 7))
    for ci, cond in enumerate(CONDITIONS):
        recs  = [rec for rec in respondent_data if rec['cond'] == cond]
        n     = len(recs)
        means, lo_errs, hi_errs = [], [], []
        for qid in QIDS:
            vals = np.array([rec[qid][metric] for rec in recs]) * scale
            m    = vals.mean() if n else 0
            lo, hi = bootstrap_ci(vals, rng=rng_boot)
            means.append(m); lo_errs.append(lo); hi_errs.append(hi)
        ax.bar(x + offsets[ci], means, width=bar_w * 0.9,
               color=COND_COLORS[cond], label=f'{COND_LABELS[cond]} (N={n})', zorder=2)
        ax.errorbar(x + offsets[ci], means, yerr=[lo_errs, hi_errs],
                    fmt='none', ecolor='#444', elinewidth=1, capsize=3, capthick=1, zorder=3)
    ax.set_xticks(x)
    ax.set_xticklabels([f'{qid}{"*" if qid in PARAPHRASE else ""}' for qid in QIDS], fontsize=10)
    ax.set_xlim(-0.6, len(QIDS) - 0.4)
    ax.set_ylabel(ylabel, fontsize=10)
    ax.set_ylim(*ylim)
    ax.set_title(f'{title}\nerror bars = 95% bootstrap CI  |  * = paraphrase', fontsize=10)
    ax.legend(fontsize=9, ncol=n_cond, loc='upper right')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    plt.tight_layout()
    fig.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved {fname}')

bar_plot('aon',     '% correct (all-or-nothing)',  'per_question_accuracy.png', (0, 110),    'Per-question accuracy by condition',        scale=100)
bar_plot('hamming', 'Mean per-statement accuracy', 'per_question_hamming.png',  (0.4, 1.05), 'Per-question Hamming accuracy by condition', scale=1)

In [ ]:
# ── Per-question Hamming accuracy by condition ────────────────────────────────
# Same layout as all-or-nothing accuracy, but y = mean per-statement accuracy.
# Error bars = 95% bootstrap CI.

x = np.arange(len(ANALYSIS_QIDS))

fig, ax = plt.subplots(figsize=(16, 5))

for i, qid in enumerate(ANALYSIS_QIDS):
    if qid in PRIMARY:
        ax.axvspan(i - 0.5, i + 0.5, color='#f5f5f5', zorder=0)

for ci, cond in enumerate(CONDITIONS):
    recs  = [rec for rec in respondent_data if rec['cond'] == cond]
    n     = len(recs)
    means, lo_errs, hi_errs = [], [], []
    for qid in ANALYSIS_QIDS:
        vals = [rec[qid]['hamming'] for rec in recs]
        m    = np.mean(vals) if n else 0
        lo, hi = bootstrap_ci(np.array(vals), rng=rng_boot)
        means.append(m); lo_errs.append(lo); hi_errs.append(hi)

    ax.bar(x + offsets[ci], means, width=bar_w * 0.92,
           color=COND_COLORS[cond], label=f'{COND_LABELS[cond]} (N={n})', zorder=2)
    ax.errorbar(x + offsets[ci], means,
                yerr=[lo_errs, hi_errs],
                fmt='none', ecolor='#333', elinewidth=0.9, capsize=2.5, capthick=0.9, zorder=3)

ax.set_xticks(x)
ax.set_xticklabels([f'{qid}{"*" if qid in PARAPHRASE else ""}' for qid in ANALYSIS_QIDS], fontsize=9)
ax.set_ylabel('Mean per-statement accuracy', fontsize=9)
ax.set_ylim(0.4, 1.05)
ax.set_title(
    'Per-question Hamming accuracy by condition  |  error bars = 95% bootstrap CI  |  '
    '* = paraphrase  |  grey = primary subset',
    fontsize=10
)
ax.legend(fontsize=8, ncol=n_cond)

plt.tight_layout()
fig.savefig('per_question_hamming.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved per_question_hamming.png')

In [ ]:
# ── Summary table ─────────────────────────────────────────────────────────────
print(f'{"Condition":<22} {"N":>4}  {"Mean AoN /10":>12}  {"Mean Ham":>9}')
print('-' * 52)
for cond in CONDITIONS:
    recs = [rec for rec in respondent_data if rec['cond'] == cond]
    n    = len(recs)
    if not recs:
        continue
    mean_aon = np.mean([r['total_aon']     for r in recs])
    mean_ham = np.mean([r['total_hamming'] for r in recs])
    print(f'{COND_LABELS[cond]:<22} {n:>4}  {mean_aon:>12.2f}  {mean_ham:>9.3f}')

print(f'\nAll-or-nothing scored over all {len(QIDS)} questions.')
print('AoN = all-or-nothing count correct; Ham = mean per-statement accuracy')

In [ ]:
# ── Reported difficulty by condition ─────────────────────────────────────────
# Q34 = reading difficulty (0=easy, 10=hard)
# Q23  = question difficulty (0=easy, 10=hard)
# Box plot + individual points, one panel per measure.

diff_cols = {
    'Reading difficulty (Q34)':   'Q34',
    'Question difficulty (Q23)': 'Q23',
}

# Pull raw values alongside condition
diff_data = {label: {cond: [] for cond in CONDITIONS} for label in diff_cols}
for r in data_rows:
    cond = r['assigned_doc']
    for label, col in diff_cols.items():
        val = r.get(col, '').strip()
        if val:
            try:
                diff_data[label][cond].append(float(val))
            except ValueError:
                pass

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
rng = np.random.default_rng(42)

for ax, (label, col) in zip(axes, diff_cols.items()):
    for ci, cond in enumerate(CONDITIONS):
        vals = np.array(diff_data[label][cond])
        if len(vals) == 0:
            continue
        color = COND_COLORS[cond]

        # Box plot
        bp = ax.boxplot(vals, positions=[ci], widths=0.5,
                        patch_artist=True, zorder=2,
                        boxprops=dict(facecolor=color, alpha=0.35, linewidth=1.2),
                        medianprops=dict(color=color, linewidth=2),
                        whiskerprops=dict(color=color, linewidth=1.2),
                        capprops=dict(color=color, linewidth=1.2),
                        flierprops=dict(marker='', linestyle='none'))

        # Jittered individual points
        jitter = rng.uniform(-0.18, 0.18, size=len(vals))
        ax.scatter(ci + jitter, vals, color=color, alpha=0.7, s=25, zorder=3)

        # Mean marker
        ax.scatter(ci, vals.mean(), marker='D', color=color, s=50,
                   edgecolors='white', linewidths=0.8, zorder=4)

    ax.set_xticks(range(len(CONDITIONS)))
    ax.set_xticklabels([COND_LABELS[c] for c in CONDITIONS], fontsize=9)
    ax.set_title(label, fontsize=10)
    ax.set_ylabel('Difficulty (0=easy, 10=hard)', fontsize=9)
    ax.set_ylim(-0.5, 10.5)
    ax.set_yticks(range(11))
    ax.tick_params(axis='y', labelsize=8)

    # Annotate means
    for ci, cond in enumerate(CONDITIONS):
        vals = np.array(diff_data[label][cond])
        if len(vals):
            ax.text(ci, vals.mean() + 0.45, f'{vals.mean():.1f}',
                    ha='center', fontsize=7.5, color=COND_COLORS[cond], fontweight='bold')

plt.suptitle('Reported difficulty by condition  |  diamond=mean  |  box=IQR',
             fontsize=10, y=1.02)
plt.tight_layout()
fig.savefig('reported_difficulty.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved reported_difficulty.png')

In [ ]:
# ── Per-respondent accuracy (all-or-nothing) ─────────────────────────────────
# One bar per respondent, sorted descending, colored by condition.

sorted_recs = sorted(respondent_data, key=lambda r: -r['total_aon'])
accs   = [r['total_aon'] / len(QIDS) * 100 for r in sorted_recs]
colors = [COND_COLORS[r['cond']] for r in sorted_recs]

fig, ax = plt.subplots(figsize=(16, 5))
bars = ax.bar(range(len(sorted_recs)), accs, color=colors, edgecolor='white', linewidth=0.4)

# Legend
from matplotlib.patches import Patch
legend_handles = [Patch(facecolor=COND_COLORS[c], label=COND_LABELS[c]) for c in CONDITIONS]
ax.legend(handles=legend_handles, fontsize=8, ncol=len(CONDITIONS), loc='upper right')

ax.set_xlabel('Respondent (sorted by accuracy)', fontsize=9)
ax.set_ylabel('% correct (all-or-nothing)', fontsize=9)
ax.set_ylim(0, 110)
ax.set_xticks([])
ax.set_title(f'Per-respondent accuracy  (N={len(sorted_recs)}, {len(QIDS)} questions)', fontsize=10)
ax.axhline(np.mean(accs), color='black', linestyle='--', linewidth=1, label=f'mean={np.mean(accs):.0f}%')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Annotate score on each bar
for i, (bar, acc) in enumerate(zip(bars, accs)):
    ax.text(bar.get_x() + bar.get_width()/2, acc + 1, f'{acc:.0f}',
            ha='center', va='bottom', fontsize=6, rotation=90)

plt.tight_layout()
fig.savefig('per_respondent_accuracy.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved per_respondent_accuracy.png')